# STAC Clip — STAC in, STAC out

Demonstrates a full STAC round-trip on the MAAP:

1. **Search** the MAAP STAC catalog for a granule with [`pystac-client`](https://pystac-client.readthedocs.io/) (the Python equivalent of the R [`rstac`](https://docs.maap-project.org/en/ogc/technical_tutorials/working_with_r/maap_stac_r.html) tutorial).
2. **Download** the granule's raster asset with [`maap-py`](https://github.com/MAAP-Project/maap-py).
3. **Clip** the raster to a bounding box with [`rasterio`](https://rasterio.readthedocs.io/).
4. **Generate STAC output** — a STAC Item describing the clip — with [`rio-stac`](https://developmentseed.org/rio-stac/).

The notebook takes a granule (collection + item id) and a raster **asset name** within that granule as input, and writes a clipped Cloud-Optimized GeoTIFF plus a STAC Item into `output/` so an OGC workflow can collect it.

In [ ]:
# Papermill parameters cell -- values here are overridden at execution time.
collection_id = "ESACCI_Biomass_L4_AGB_V4_100m"
granule_id = "S50W080_ESACCI-BIOMASS-L4-AGB-MERGED-100m-2020-fv4.0"
asset_name = "estimates"
bbox = "-78 -58 -76 -56"  # MINX MINY MAXX MAXY, EPSG:4326
output_file = "clipped.tif"
stac_url = "https://stac.maap-project.org/"
maap_host = "api.maap-project.org"

In [ ]:
import json
import os

import pystac
import rasterio
import rasterio.shutil
from maap.maap import MAAP
from pystac_client import Client
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from rio_stac.stac import create_stac_item

OUTPUT_DIR = "output"
DOWNLOAD_DIR = "downloads"

## 1. Search the STAC catalog

`maap-py` searches CMR, not STAC, so the STAC search is done with `pystac-client` against the MAAP
STAC endpoint. We look up the requested granule (a STAC Item) within its collection and resolve the
download URL (`href`) of the requested raster asset.

In [ ]:
def search_stac(stac_url, collection_id, granule_id, asset_name):
    """Find ``granule_id`` in ``collection_id`` and return (item, asset_href)."""
    catalog = Client.open(stac_url)
    search = catalog.search(collections=[collection_id], ids=[granule_id])
    items = list(search.items())
    if not items:
        raise LookupError(
            f"Granule '{granule_id}' not found in collection '{collection_id}'."
        )
    item = items[0]
    if asset_name not in item.assets:
        raise KeyError(
            f"Asset '{asset_name}' not found. Available: {sorted(item.assets)}"
        )
    href = item.assets[asset_name].href
    print(f"Found item {item.id!r}; asset {asset_name!r} -> {href}")
    return item, href

## 2. Download the granule asset with maap-py

`MAAP.downloadGranule` handles both `s3://` (via boto3, using ambient AWS credentials on the MAAP
platform) and `https` URLs, and returns the local path to the downloaded file.

In [ ]:
def download_asset(href, maap_host, dest_dir=DOWNLOAD_DIR):
    """Download ``href`` with maap-py and return the local file path."""
    os.makedirs(dest_dir, exist_ok=True)
    maap = MAAP(maap_host=maap_host)
    local_path = maap.downloadGranule(
        href, destination_path=dest_dir, overwrite=True
    )
    print(f"Downloaded asset to {local_path}")
    return local_path

## 3. Clip the raster to the bounding box

The bbox is supplied in EPSG:4326 and reprojected to the raster's CRS. A windowed read pulls only
the clipped region off the local file, which is then written out as a valid Cloud-Optimized GeoTIFF.

In [ ]:
def clip_asset(local_path, bbox, output_path):
    """Window-read ``local_path`` to ``bbox`` (lon/lat) and write it as a COG."""
    with rasterio.open(local_path) as src:
        # The bbox is in EPSG:4326; reproject it to the raster's CRS.
        dst_bounds = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*dst_bounds, transform=src.transform)
        window = window.round_offsets().round_lengths()

        if window.width <= 0 or window.height <= 0:
            raise ValueError(
                f"bbox {bbox} does not intersect the raster bounds {src.bounds}."
            )

        data = src.read(window=window)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=data.shape[1],
            width=data.shape[2],
            transform=transform,
            tiled=False,
        )
        # Drop any source block sizes that no longer fit the clipped raster.
        for key in ("blockxsize", "blockysize"):
            profile.pop(key, None)

        # Write to an in-memory GeoTIFF, then CreateCopy it to a valid COG.
        with MemoryFile() as mem:
            with mem.open(**profile) as tmp:
                tmp.write(data)
                rasterio.shutil.copy(
                    tmp, output_path, driver="COG", overwrite=True
                )
    print(f"Wrote clipped COG to {output_path}")
    return output_path

## 4. Generate the STAC output with rio-stac

`rio_stac.create_stac_item` inspects the clipped COG and builds a STAC Item, including the
projection (`proj:`) and raster (`raster:bands`) extension metadata. The asset `href` is written as
the output filename so the Item and the COG sit side by side in `output/`.

In [ ]:
def write_stac_output(
    output_path, output_file, src_item, asset_name, collection_id, output_dir=OUTPUT_DIR
):
    """Build a STAC Item for the clipped COG and write it to ``output_dir``."""
    out_id = f"{src_item.id}-clip"
    item = create_stac_item(
        source=output_path,
        input_datetime=src_item.datetime,
        id=out_id,
        asset_name=asset_name,
        asset_href=output_file,  # sibling of the item JSON in the output dir
        asset_media_type=pystac.MediaType.COG,
        asset_roles=["data"],
        collection=collection_id,
        with_proj=True,
        with_raster=True,
        properties={"derived_from": src_item.id},
    )
    item_path = os.path.join(output_dir, f"{out_id}.json")
    with open(item_path, "w") as f:
        json.dump(item.to_dict(), f, indent=2)
    print(f"Wrote STAC Item to {item_path}")
    return item_path

## Run the workflow

In [ ]:
bbox_values = [float(v) for v in bbox.split()]
if len(bbox_values) != 4:
    raise ValueError("bbox must have four values: 'MINX MINY MAXX MAXY'")

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, output_file)

src_item, href = search_stac(stac_url, collection_id, granule_id, asset_name)
local_path = download_asset(href, maap_host)
clip_asset(local_path, bbox_values, output_path)
item_path = write_stac_output(
    output_path, output_file, src_item, asset_name, collection_id
)

print(f"\nDone. Clipped COG: {output_path}\nSTAC Item:   {item_path}")